# Calculo del divisor resistivo

Este notebook presenta el cálculo de la viabilidad de los componentes elegidos para contruir un divisor resistivo para el método 0.
El circuito analizado es el siguiente:

<img src="./circuito.png">

In [1]:
import numpy as np

In [2]:
# Exactitud de vuelta (lo que podemos llegar a controlar a pulso)
EX_GIRO = 3 # grados 

# Esta es la precisión con la cual vamos a hacer la búsqueda 
# de la posición de los potenciómetros 
# (parche progrmático para evitar una búsqueda exhaustiva de
# un valor que será aproximado en la práctica)
div_tol = 0.00001

# Corriente de la bateria maxima, esto es lo máximo
# que permitiremos que se descargue la bateria
I_bat_max = 1e-3 # A
# Tensión de la bateria (celda 9v)
V_bat = 9.0 # V
# Tensión a calibrar (regulador 7805)
V_cal = 5.0 # V

# Resistencias fijas elejidas.
# Recuerden que esto es un problema de diseño
# Los valores propuestos responden a los criterios
# de corriente de la batería y precisión del preset Rp1
R1 = 4.7e3 # ohm
Rp1 = 500 # ohm 
R2 = 4.7e3 # ohm
Rp2 = 1e3 # ohm 
# Numero vueltas Rp1
N_turn_Rp1 = 25
range_Rp1 = N_turn_Rp1*360.0
# Numero vueltas Rp2
N_turn_Rp2 = 1
range_Rp2 = N_turn_Rp2*360.0



# Función del divisor

Esta función representa el cálculo del dividor, como se utilizará en repetidas ocaciones la declaramos como función para evitar problemas de escritura.

In [3]:
def calcular_divisor(R1_in, Rp1_in, R2_in, Rp2_in, frac_rp1_in, frac_rp2_in):
    
    # Resistencia total que ve la bateria, compuesta por todas
    # las resistencias y la porción del preset Rp2 que no se 
    # encuentra derivado a tierra.
    R_tot = (R1_in + Rp1_in + R2_in + Rp2_in*( frac_rp2_in )  )
    
    # Resistencia que queda sobre el voltimetro. Compuesta por
    # la R2, la porción de Rp2 en uso y la porción de Rp1 debajo
    # del punto de contacto.
    R_div = ( Rp1_in*( frac_rp1_in ) + R2_in + Rp2_in*( frac_rp2_in ) )
    
    # Devolvemos el valor del divisor
    return  (R_div / R_tot)

# Testeamos los valores elegidos

Estos son los parametros de disño que debemos respetar.

In [4]:
# Valor de resistencia a respetar
R_tot_min = V_bat/I_bat_max
print("Minima resistencia que debe ver la bateria:\n\tR_tot_min = %0.2f [kOhm]"%(R_tot_min/1000.0))

# Valor de divisor deseado
div_val = V_cal/V_bat
print("Valor de divisor:\n\tdiv_val = %0.2f [-]"%div_val)

Minima resistencia que debe ver la bateria:
	R_tot_min = 9.00 [kOhm]
Valor de divisor:
	div_val = 0.56 [-]


### Valores extremo de los potenciometros

Primero debemos confirmar que los valores elegidos de resistencias pueden alcanzar los valores de diseño.
Para ello calculamos las posiciones de máxima y mínima exursión de los potenciometros. 

In [5]:
# Ambos al máximo
frac_rp1 = 1
frac_rp2 = 1
div_1_1 = calcular_divisor(R1, Rp1, R2, Rp2, frac_rp1, frac_rp2)
R_tot_d_1_1 = R1 + Rp1 + R2 + Rp2*( frac_rp2 )
# Ambos al mínimo
frac_rp1 = 0
frac_rp2 = 0
div_0_0 = calcular_divisor(R1, Rp1, R2, Rp2, frac_rp1, frac_rp2)
R_tot_d_0_0 = R1 + Rp1 + R2 + Rp2*( frac_rp2 )
# Combinaciones
frac_rp1 = 0
frac_rp2 = 1
div_0_1 = calcular_divisor(R1, Rp1, R2, Rp2, frac_rp1, frac_rp2)
R_tot_d_0_1 = R1 + Rp1 + R2 + Rp2*( frac_rp2 )
frac_rp1 = 1
frac_rp2 = 0
div_1_0 = calcular_divisor(R1, Rp1, R2, Rp2, frac_rp1, frac_rp2)
R_tot_d_1_0 = R1 + Rp1 + R2 + Rp2*( frac_rp2 )

# Buscamos el minimo y máximo del divisor y valor de resistencia
div_min = np.min([div_1_1,div_0_0,div_0_1,div_1_0])
div_min_idx = np.argmin([div_1_1,div_0_0,div_0_1,div_1_0])
div_max = np.max([div_1_1,div_0_0,div_0_1,div_1_0])
div_max_idx = np.argmax([div_1_1,div_0_0,div_0_1,div_1_0])

R_tot_d_min = np.min([R_tot_d_1_1,R_tot_d_0_0,R_tot_d_0_1,R_tot_d_1_0])
R_tot_d_min_idx = np.argmin([R_tot_d_1_1,R_tot_d_0_0,R_tot_d_0_1,R_tot_d_1_0])
R_tot_d_max = np.max([R_tot_d_1_1,R_tot_d_0_0,R_tot_d_0_1,R_tot_d_1_0])
R_tot_d_max_idx = np.argmax([R_tot_d_1_1,R_tot_d_0_0,R_tot_d_0_1,R_tot_d_1_0])


# Informamos los limites
configuraciones_limite = ['Rp1 = 1, Rp2 = 1',
                         'Rp1 = 0, Rp2 = 0',
                         'Rp1 = 0, Rp2 = 1',
                         'Rp1 = 1, Rp2 = 0']

print("Divisor máximo:")
print("\tConfiguracion:\t\t%s"%configuraciones_limite[div_max_idx])
print("\tValor del divisor: \tdiv_val = %0.2f [-]"%div_max)
print("\tValor de Resistencia:\tR_tot = %0.2f [kOhm]"%(R_tot_d_max/1e3))
print("-----------------------------------------------------------------------")

print("Divisor mínimo:")
print("\tConfiguracion:\t\t%s"%configuraciones_limite[div_min_idx])
print("\tValor del divisor:\tdiv_val = %0.2f [-]"%div_min)
print("\tValor de Resistencia:\tR_tot = %0.2f [kOhm]"%(R_tot_d_min/1e3))



Divisor máximo:
	Configuracion:		Rp1 = 1, Rp2 = 1
	Valor del divisor: 	div_val = 0.57 [-]
	Valor de Resistencia:	R_tot = 10.90 [kOhm]
-----------------------------------------------------------------------
Divisor mínimo:
	Configuracion:		Rp1 = 0, Rp2 = 0
	Valor del divisor:	div_val = 0.47 [-]
	Valor de Resistencia:	R_tot = 9.90 [kOhm]


Ahora debemos checkear que los limites incluyan los valores de diseño:

In [6]:
if (div_min > div_val or div_max < div_val or R_tot_d_max < R_tot_min or R_tot_d_min < R_tot_min):
    
    print("LOS VALORES SELECCIONADOS NO CUMPLEN CON LOS REQUISITOS")
    
    if (R_tot_d_max < R_tot_min or R_tot_d_min < R_tot_min):
        print("\tResistencia total muy pequeña.")
    if (div_min > div_val or div_max < div_val):
        print("\tValores de división fuera de rango.")
else:
    print("Los valores seleccionados cumplen con los límites.")

Los valores seleccionados cumplen con los límites.


### Posición ideal de los presets

En esta sección exploraremos si existe una configuración donde los present se encuentran en zonas de trabajo deseable y que cumplan con los críterios de disño seleccionado.
Las zonas de trabajo deseable son el centro del potenciometro 1, donde nos encontramos lejos de los efectos de borde del mismo. En el potenciometro 2, la zona deseada es a plena escala ya que deseamos que la bateria vea la mayor carga posible.

In [7]:
# Primero seteamos el potenciometro 1 a mitad de escala
mid_range_Rp1 = (range_Rp1/2.0)
mid_range_steps_Rp1 = int(mid_range_Rp1/EX_GIRO)
# y el potenciometro 2 del fin de escala hacia el cero
range_steps_Rp2 = np.arange(range_Rp2,0,-EX_GIRO)

# Iniciamos la busqueda
found = False
step_found_rp1 = 0
step_found_rp2 = 0
div_found = 0
# para cada valor del centro del potenciometro
for i in range(0, mid_range_steps_Rp1):

    # Si lo encontre salgo del loop
    if found:
        break
        
    # Checkeo todos los valores del potenciometro 2
    for step_use_Rp2 in range_steps_Rp2:
        
        # Testeo para cada lado, ya que el primer
        # loop solo recorre la mitad de la cantidad
        # de pasos del potenciometro 1.
        # Esto es así porque lo exploro desde el centro.
        
        # Calculo la fraccion de Rp1 (Lado negativo)
        step_use_Rp1 = mid_range_Rp1 - (EX_GIRO*i)
        frac_rp1 = step_use_Rp1/range_Rp1
        # Calculo la fraccion de Rp2 (paso actual)
        frac_rp2= step_use_Rp2/range_Rp2
        # Calculo el valor del divisor
        div_act = calcular_divisor(R1, Rp1, R2, Rp2, frac_rp1, frac_rp2)
        # Checkeo que el divisor actual este dentro de
        # la tolerancia seleccionada para el valor
        # de divisor dado por el diseño
        if div_act < div_val*(1+div_tol) and div_act > div_val*(1-div_tol) :
            print("Valor encontrado")
            div_found = div_act
            step_found_rp1 = step_use_Rp1
            step_found_rp2 = step_use_Rp2
            found = True
            break
  
        # --------------- Lo mismo para el otro lado.
        # Calculo la fraccion de Rp1 (Lado positivo)
        step_use_Rp1 = mid_range_Rp1 + (EX_GIRO*i)
        frac_rp1 = step_use_Rp1/range_Rp1
        # Calculo la fraccion de Rp2 (paso actual)
        frac_rp2= step_use_Rp2/range_Rp2
        # Calculo el valor del divisor
        div_act = calcular_divisor(R1, Rp1, R2, Rp2, frac_rp1, frac_rp2)
        # Checkeo que el divisor actual este dentro de
        # la tolerancia seleccionada para el valor
        # de divisor dado por el diseño
        if div_act < div_val*(1+div_tol) and div_act > div_val*(1-div_tol):
            print("Valor encontrado")
            div_found = div_act
            step_found_rp1 = step_use_Rp1
            step_found_rp2 = step_use_Rp2
            found = True
            break
            
    # Una vez terminado el ciclo y si no lo encontré, vuelvo
    # a recorrer todo el potenciometro Rp2, pero ampliando
    # el corriemiento del potenciometro Rp1
    
# Si llegue acá debería haber encontrado un punto
if not found:
    print("Los valores seleccionados no cumplen con la tolerancia especificada o con los valores de diseño (¿fueron checkeados antes de correr esta sección?).")

Valor encontrado


In [8]:
# Informo el punto de trabajo encontrado

print("Punto de trabajo:")
print("\tdiv_act = %0.2f [-]"%div_act)
print("\tstep_Rp1 = %0.2f [º]"%step_found_rp1)
print("\tstep_Rp2 = %0.2f [º]"%step_found_rp2)

# Calculamos las vueltas
Vueltas_Rp1 = step_found_rp1/360.0
Vueltas_Rp2 = step_found_rp2/360.0
print("\tVueltas_Rp1 = %0.2f [-]"%Vueltas_Rp1)
print("\tVueltas_Rp2 = %0.2f [-]"%Vueltas_Rp2)
# Corriente sobre la beteria
frac_rp2 = (step_found_rp2)/range_Rp2
R_tot = (R1 + Rp1 + R2 + Rp2*( frac_rp2 )  )
I_bat = V_bat/R_tot
print("Corriente sobre la bateria: I_bat = %0.5f [mA]"%(I_bat*1e3))

# Checkeamos los limites
if N_turn_Rp1 == Vueltas_Rp1 or Vueltas_Rp1 == 0:
    print("\n¡CUIDADO! : Potenciometro Rp1 en el límite de excursión")
if N_turn_Rp2 == Vueltas_Rp2 or Vueltas_Rp1 == 0:
    print("\n¡CUIDADO! : Potenciometro Rp2 en el límite de excursión")

Punto de trabajo:
	div_act = 0.56 [-]
	step_Rp1 = 6399.00 [º]
	step_Rp2 = 360.00 [º]
	Vueltas_Rp1 = 17.77 [-]
	Vueltas_Rp2 = 1.00 [-]
Corriente sobre la bateria: I_bat = 0.82569 [mA]

¡CUIDADO! : Potenciometro Rp2 en el límite de excursión


### Testeamos la precision del paso

Finalmente comprobamos si el potenciometro Rp1 tiene un paso acorde a la tensión a medir. Esto ayudara a setear el cero.
Si queremos medir decenas de mV, es deseable que el paso sea unidades de mV o menor.
Al igual que antes, este es un valor que depende de la combinatoria de de los dos potenciometros, es decir, hay cuatro valores limites de los cuales nos interesan los 2 extremos.

In [9]:


# Calculamos el valor de fracción en la posicion dada, 
# pero con un error de la mitad de la precisión.

# Combinatoria
frac_rp1 = (step_found_rp1+(EX_GIRO/2.0))/range_Rp1
frac_rp2 = (step_found_rp2+(EX_GIRO/2.0))/range_Rp2
div_act_step_p_p = calcular_divisor(R1, Rp1, R2, Rp2, frac_rp1, frac_rp2)
frac_rp1 = (step_found_rp1-(EX_GIRO/2.0))/range_Rp1
frac_rp2 = (step_found_rp2-(EX_GIRO/2.0))/range_Rp2
div_act_step_n_n = calcular_divisor(R1, Rp1, R2, Rp2, frac_rp1, frac_rp2)
frac_rp1 = (step_found_rp1+(EX_GIRO/2.0))/range_Rp1
frac_rp2 = (step_found_rp2-(EX_GIRO/2.0))/range_Rp2
div_act_step_p_n = calcular_divisor(R1, Rp1, R2, Rp2, frac_rp1, frac_rp2)
frac_rp1 = (step_found_rp1-(EX_GIRO/2.0))/range_Rp1
frac_rp2 = (step_found_rp2+(EX_GIRO/2.0))/range_Rp2
div_act_step_n_p = calcular_divisor(R1, Rp1, R2, Rp2, frac_rp1, frac_rp2)

# Calculamos el valor mostroado en el volímetro
V_tester_p_p = V_cal - div_act_step_p_p * V_bat
V_tester_n_n = V_cal - div_act_step_n_n * V_bat
V_tester_p_n = V_cal - div_act_step_p_n * V_bat
V_tester_n_p = V_cal - div_act_step_n_p * V_bat

# Buscamos los extremos
V_tester_min = np.max(np.abs([V_tester_p_p, V_tester_n_n,V_tester_p_n, V_tester_n_p]))

# Informamos 
print("El módulo del error máximo por paso:")
print("\tV_err_max = |%0.2f| [mV]"%(V_tester_min*1e3))



El módulo del error máximo por paso:
	V_err_max = |1.64| [mV]
